# 08 — Strategy Report

**Purpose:** Aggregate all backtested strategy candidates into a ranked report table.

Each candidate receives one of four classification labels:

| Status | Meaning |
|--------|---------|
| `EA_CANDIDATE` | Robust across all folds, passes all filters → ready for paper trading then EA |
| `PAPER_TEST` | Promising but needs more live observation before building an EA |
| `NEEDS_MORE_DATA` | Insufficient trades or short history — cannot conclude yet |
| `REJECTED` | Fails one or more minimum criteria |

**Minimum criteria for EA_CANDIDATE:**
- Profit factor ≥ 1.5 in aggregated OOS
- Profit factor ≥ 1.2 in ≥ 4/5 walk-forward folds
- Win rate ≥ 40%
- ≥ 30 trades in OOS period
- Max drawdown ≤ 20% of gross profit
- Sharpe ≥ 0.5
- No single month with net P&L < −3× average monthly P&L

**Inputs:** All `reports/backtest_*.csv` files  
**Outputs:** `reports/strategy_candidates.csv`, `reports/strategy_candidates.html`

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from src.utils.config import load_config, get_symbol, get_paths, get_backtest_params

cfg      = load_config()
paths    = get_paths(cfg, "..")
symbol   = get_symbol(cfg)
bt_cfg   = get_backtest_params(cfg)
reports  = paths["reports"]

MIN_TRADES     = int(bt_cfg.get("min_trades", 30))
MIN_PF         = 1.5
MIN_PF_FOLDS   = 1.2
MIN_WR         = 0.40
MIN_SHARPE     = 0.5
MAX_DD_RATIO   = 0.20  # max_dd / gross_profit

print(f"Classification thresholds:")
print(f"  Min trades       : {MIN_TRADES}")
print(f"  Min profit factor: {MIN_PF} (OOS), {MIN_PF_FOLDS} (per fold)")
print(f"  Min win rate     : {MIN_WR*100:.0f}%")
print(f"  Min Sharpe       : {MIN_SHARPE}")
print(f"  Max DD ratio     : {MAX_DD_RATIO*100:.0f}%")

## 1. Scan backtest result files

In [ ]:
# Collect all walk-forward summary files
wf_files   = sorted(reports.glob("backtest_walkforward_*.csv"))
trade_files = sorted(reports.glob("backtest_trades_*.csv"))
monthly_files = sorted(reports.glob("backtest_monthly_*.csv"))

print(f"Found {len(wf_files)} walk-forward result files:")
for f in wf_files:
    print(f"  {f.name}")

## 2. Classify each strategy candidate

In [ ]:
def classify_strategy(
    fold_df: pd.DataFrame,
    trade_df: pd.DataFrame | None,
    monthly_df: pd.DataFrame | None,
) -> tuple[str, list[str]]:
    """Apply classification rules and return (status, reasons)."""
    reasons: list[str] = []
    
    # Aggregate OOS metrics
    n_trades   = fold_df["n_trades"].sum() if "n_trades" in fold_df else 0
    net_profit = fold_df["net_profit"].sum() if "net_profit" in fold_df else 0
    pf_folds   = fold_df["profit_factor"].dropna() if "profit_factor" in fold_df else pd.Series()
    pf_oos     = pf_folds.mean() if not pf_folds.empty else 0
    wr_oos     = fold_df["win_rate"].mean() if "win_rate" in fold_df else 0
    sharpe_oos = fold_df["sharpe"].mean() if "sharpe" in fold_df else 0
    max_dd     = fold_df["max_drawdown"].min() if "max_drawdown" in fold_df else 0
    gross_p    = fold_df["gross_profit"].sum() if "gross_profit" in fold_df else 1

    # Check each criterion
    if n_trades < MIN_TRADES:
        reasons.append(f"Insufficient trades: {n_trades} < {MIN_TRADES}")
        return "NEEDS_MORE_DATA", reasons

    if pf_oos < MIN_PF:
        reasons.append(f"PF too low: {pf_oos:.2f} < {MIN_PF}")

    if not pf_folds.empty:
        n_passing_folds = (pf_folds >= MIN_PF_FOLDS).sum()
        if n_passing_folds < len(pf_folds) * 0.8:
            reasons.append(
                f"Only {n_passing_folds}/{len(pf_folds)} folds have PF≥{MIN_PF_FOLDS}"
            )

    if wr_oos < MIN_WR:
        reasons.append(f"Win rate too low: {wr_oos*100:.1f}% < {MIN_WR*100:.0f}%")

    if sharpe_oos < MIN_SHARPE:
        reasons.append(f"Sharpe too low: {sharpe_oos:.2f} < {MIN_SHARPE}")

    dd_ratio = abs(max_dd) / max(gross_p, 1)
    if dd_ratio > MAX_DD_RATIO:
        reasons.append(f"DD ratio too high: {dd_ratio*100:.1f}% > {MAX_DD_RATIO*100:.0f}%")

    # Monthly consistency check
    if monthly_df is not None and not monthly_df.empty and "net_pnl" in monthly_df:
        avg_monthly = monthly_df["net_pnl"].mean()
        worst_month = monthly_df["net_pnl"].min()
        if worst_month < -3 * abs(avg_monthly):
            reasons.append(
                f"Worst month ({worst_month:.0f}) < -3× avg ({avg_monthly:.0f})"
            )

    if not reasons:
        return "EA_CANDIDATE", ["Passes all criteria"]
    elif len(reasons) == 1 and ("PF" in reasons[0] or "Sharpe" in reasons[0]):
        return "PAPER_TEST", reasons
    else:
        return "REJECTED", reasons


print("Classification function defined.")

In [ ]:
candidate_rows = []

for wf_file in wf_files:
    # Parse strategy info from filename: backtest_walkforward_{model}_{key}.csv
    stem   = wf_file.stem  # e.g. backtest_walkforward_random_forest_tp2.0_sl1.0_h30
    parts  = stem.replace("backtest_walkforward_", "").split("_")
    
    fold_df = pd.read_csv(wf_file)
    
    # Load corresponding trade and monthly files
    trade_stem   = wf_file.name.replace("walkforward", "trades")
    monthly_stem = wf_file.name.replace("walkforward", "monthly")
    
    trade_df   = pd.read_csv(reports / trade_stem)   if (reports / trade_stem).exists()   else None
    monthly_df = pd.read_csv(reports / monthly_stem) if (reports / monthly_stem).exists() else None

    status, reasons = classify_strategy(fold_df, trade_df, monthly_df)

    row = {
        "strategy_id":    stem,
        "symbol":         symbol,
        "side":           "long",  # TODO: extend for short
        "n_trades":       int(fold_df["n_trades"].sum()) if "n_trades" in fold_df else 0,
        "net_profit":     round(fold_df["net_profit"].sum(), 2) if "net_profit" in fold_df else 0,
        "profit_factor":  round(fold_df["profit_factor"].mean(), 3) if "profit_factor" in fold_df else 0,
        "win_rate_pct":   round(fold_df["win_rate"].mean() * 100, 1) if "win_rate" in fold_df else 0,
        "expectancy":     round(fold_df["expectancy"].mean(), 2) if "expectancy" in fold_df else 0,
        "avg_r":          round(fold_df["avg_r"].mean(), 3) if "avg_r" in fold_df else 0,
        "sharpe":         round(fold_df["sharpe"].mean(), 3) if "sharpe" in fold_df else 0,
        "sortino":        round(fold_df["sortino"].mean(), 3) if "sortino" in fold_df else 0,
        "max_drawdown":   round(fold_df["max_drawdown"].min(), 2) if "max_drawdown" in fold_df else 0,
        "n_wf_folds":     len(fold_df),
        "wf_pf_folds":    str(fold_df["profit_factor"].round(2).tolist()) if "profit_factor" in fold_df else "",
        "status":         status,
        "reasons":        " | ".join(reasons),
    }
    candidate_rows.append(row)
    print(f"  {status:17s} — {stem}")
    for r in reasons:
        print(f"               • {r}")

candidates_df = pd.DataFrame(candidate_rows)
print(f"\nTotal candidates evaluated: {len(candidates_df)}")

## 3. Ranked final table

In [ ]:
# Sort by status priority then profit factor
status_order = {"EA_CANDIDATE": 0, "PAPER_TEST": 1, "NEEDS_MORE_DATA": 2, "REJECTED": 3}
candidates_df["status_rank"] = candidates_df["status"].map(status_order)
candidates_df = candidates_df.sort_values(
    ["status_rank", "profit_factor"], ascending=[True, False]
).drop(columns=["status_rank"])

# Display with colour coding
def _style_status(val):
    colors = {
        "EA_CANDIDATE":    "background-color: #d4edda; color: #155724",
        "PAPER_TEST":      "background-color: #d1ecf1; color: #0c5460",
        "NEEDS_MORE_DATA": "background-color: #fff3cd; color: #856404",
        "REJECTED":        "background-color: #f8d7da; color: #721c24",
    }
    return colors.get(val, "")

styled = (
    candidates_df
    .style
    .applymap(_style_status, subset=["status"])
    .format({
        "net_profit":   "{:+.2f}",
        "profit_factor": "{:.3f}",
        "win_rate_pct":  "{:.1f}%",
        "expectancy":    "{:+.2f}",
        "sharpe":        "{:.3f}",
        "max_drawdown":  "{:.2f}",
    })
)
display(styled)

## 4. Export to CSV and HTML

In [ ]:
# CSV
csv_path = reports / "strategy_candidates.csv"
candidates_df.to_csv(csv_path, index=False)
print(f"CSV  saved → {csv_path}")

# HTML
html_path = reports / "strategy_candidates.html"
html_content = f"""
<!DOCTYPE html>
<html>
<head>
    <meta charset='utf-8'>
    <title>DAX Strategy Candidates — {symbol}</title>
    <style>
        body {{ font-family: Arial, sans-serif; margin: 20px; }}
        h1 {{ color: #333; }}
        table {{ border-collapse: collapse; width: 100%; font-size: 13px; }}
        th, td {{ border: 1px solid #ddd; padding: 8px; text-align: right; }}
        th {{ background-color: #4a4a4a; color: white; text-align: center; }}
        tr:nth-child(even) {{ background-color: #f9f9f9; }}
        .EA_CANDIDATE  {{ background-color: #d4edda !important; font-weight: bold; }}
        .PAPER_TEST    {{ background-color: #d1ecf1 !important; }}
        .NEEDS_MORE_DATA {{ background-color: #fff3cd !important; }}
        .REJECTED      {{ background-color: #f8d7da !important; color: #888; }}
    </style>
</head>
<body>
<h1>DAX Quant Research Lab — Strategy Candidates ({symbol})</h1>
<p>Generated: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}</p>
"""

# Build HTML table manually to apply row classes
display_cols = [
    "strategy_id", "side", "n_trades", "net_profit", "profit_factor",
    "win_rate_pct", "expectancy", "avg_r", "sharpe", "sortino",
    "max_drawdown", "n_wf_folds", "status", "reasons"
]
html_content += "<table>\n<tr>"
for col in display_cols:
    html_content += f"<th>{col}</th>"
html_content += "</tr>\n"

for _, row in candidates_df[display_cols].iterrows():
    status_class = str(row["status"])
    html_content += f'<tr class="{status_class}">'
    for col in display_cols:
        val = row[col]
        if isinstance(val, float):
            cell = f"{val:+.3f}" if col in ("net_profit", "expectancy") else f"{val:.3f}"
        else:
            cell = str(val)
        html_content += f"<td>{cell}</td>"
    html_content += "</tr>\n"

html_content += "</table>\n</body>\n</html>"

with open(html_path, "w", encoding="utf-8") as f:
    f.write(html_content)
print(f"HTML saved → {html_path}")

## 5. Final summary

In [ ]:
status_counts = candidates_df["status"].value_counts()
print("="*50)
print("STRATEGY CANDIDATE SUMMARY")
print("="*50)
for status in ["EA_CANDIDATE", "PAPER_TEST", "NEEDS_MORE_DATA", "REJECTED"]:
    n = status_counts.get(status, 0)
    print(f"  {status:20s}: {n}")

print("\nNext steps:")
ea_count = status_counts.get("EA_CANDIDATE", 0)
pt_count = status_counts.get("PAPER_TEST", 0)

if ea_count > 0:
    print(f"  → {ea_count} candidate(s) ready for paper trading")
    print("     After 3+ months of paper trading with consistent results, build MQL5 EA.")
if pt_count > 0:
    print(f"  → {pt_count} candidate(s) to monitor with more data")
if ea_count == 0 and pt_count == 0:
    print("  → No strong candidates yet. Suggestions:")
    print("     1. Try different TP/SL/horizon combinations in notebook 03")
    print("     2. Add more features or try regime-conditioned signals")
    print("     3. Collect more historical data (extend start_date in config)")
    print("     4. Try different model hyperparameters in notebook 05")